In [1]:
pip install pandas numpy scikit-learn joblib openpyxl

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip available: 22.3.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
import os
import pandas as pd
import numpy as np
import joblib

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, classification_report
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import MinMaxScaler

# ============================================================
# DATASET PATH
# ============================================================

DATASET_PATH = r"C:\Users\verni\Desktop\Ayurwell\Code\data\Ayurveda.xlsx"

# ============================================================
# MODEL SAVE DIRECTORY
# ============================================================

SAVE_DIR = r"C:\Users\verni\Desktop\Ayurwell\Code\model_files"

os.makedirs(SAVE_DIR, exist_ok=True)

# ============================================================
# LOAD DATASET
# ============================================================

print("Loading dataset...")

df = pd.read_excel(DATASET_PATH)
df.columns = df.columns.str.strip().str.lower()


print(f"Dataset Shape: {df.shape}")

# ============================================================
# TARGET COLUMN
# ============================================================

TARGET_COLUMN = "disease_english"

# ============================================================
# FEATURE COLUMNS
# ============================================================

# First 4 columns are metadata
# Actual symptom columns start from column E

exclude_cols = [
    "disease_english",
    "remedies",
    "diet",
    "lifestyle"
]

feature_columns = [
    col for col in df.columns
    if col not in exclude_cols
]

# Remove empty/unnamed columns if present
feature_columns = [
    col for col in feature_columns
    if pd.notna(col)
]
feature_columns = [
    col for col in feature_columns
    if pd.api.types.is_numeric_dtype(df[col])
]
# ============================================================
# SYMPTOM RARITY WEIGHTS (TF-IDF STYLE)
# ============================================================

symptom_rarity = {}

total_diseases = df[TARGET_COLUMN].nunique()

for col in feature_columns:

    diseases_with_symptom = (
        df[df[col] > 0][TARGET_COLUMN]
        .nunique()
    )

    rarity = np.log(
        (total_diseases + 1) /
        (diseases_with_symptom + 1)
    )

    symptom_rarity[col] = rarity

# ============================================================
# CREATE FEATURES + LABELS
# ============================================================

X = df[feature_columns].copy()

y = df[TARGET_COLUMN].copy()

# ============================================================
# CLEAN DATA
# ============================================================

# Convert all values to numeric
X = X.apply(pd.to_numeric, errors='coerce')

# Replace NaN with 0
X = X.fillna(0)

# Ensure range 0-5
X = X.clip(lower=0, upper=5)

# ============================================================
# SAVE FEATURE COLUMNS
# ============================================================

joblib.dump(
    feature_columns,
    os.path.join(SAVE_DIR, "columns.pkl")
)

print("Saved columns.pkl")

# ============================================================
# LABEL ENCODING
# ============================================================

label_encoder = LabelEncoder()

y_encoded = label_encoder.fit_transform(y)

joblib.dump(
    label_encoder,
    os.path.join(SAVE_DIR, "label_encoder.pkl")
)

print("Saved label_encoder.pkl")

scaler = MinMaxScaler()

X = scaler.fit_transform(X) 

# ============================================================
# SAVE SCALER
# ============================================================

joblib.dump(
    scaler,
    os.path.join(SAVE_DIR, "scaler.pkl")
)

print("Saved scaler.pkl")

# =========================================================
# TRAIN TEST SPLIT
# =========================================================

X_train, X_test, y_train, y_test = train_test_split(
    X_scaled,
    y_encoded,
    test_size=0.2,
    random_state=42,
)

print("\nTrain/Test Split Complete.")

print(f"Training Samples: {len(X_train)}")
print(f"Testing Samples: {len(X_test)}")

# ============================================================
# ANN MODEL
# ============================================================

print("\nInitializing ANN Model...")

model = MLPClassifier(
    hidden_layer_sizes=(32, 16),
    activation='relu',
    solver='adam',
    alpha=0.0001,
    learning_rate='adaptive',
    learning_rate_init=0.001,
    max_iter=1000,
    early_stopping=True,
    validation_fraction=0.1,
    n_iter_no_change=25,
    random_state=42,
    verbose=True
)

# ============================================================
# TRAIN MODEL
# ============================================================

print("\nTraining ANN...")

model.fit(X_train, y_train)

print("\nTraining Complete.")

# ============================================================
# EVALUATION
# ============================================================

print("\nEvaluating Model...")

y_pred = model.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)

print(f"\nAccuracy: {accuracy:.4f}")

print("\nClassification Report:\n")

print(classification_report(
    y_test,
    y_pred,
))

# ============================================================
# SAVE ANN MODEL
# ============================================================

joblib.dump(
    model,
    os.path.join(SAVE_DIR, "ann_model.pkl")
)

print("Saved ann_model.pkl")

# ============================================================
# SAVE DISEASE PROTOTYPES
# ============================================================

prototype_df = df[
    [TARGET_COLUMN] + list(feature_columns)
].copy()

# Force symptom columns numeric
prototype_df[feature_columns] = (
    prototype_df[feature_columns]
    .apply(pd.to_numeric, errors='coerce')
    .fillna(0)
    .astype(float)
)

joblib.dump(
    prototype_df,
    os.path.join(SAVE_DIR, "disease_prototypes.pkl")
)

print("Saved disease_prototypes.pkl")

# ============================================================
# HYBRID PREDICTION FUNCTION
# ============================================================

def hybrid_predict(user_symptoms_dict, top_n=5):

    # ============================================================
    # NORMALIZE USER INPUT
    # ============================================================

    normalized_input = {
        k.strip().lower(): float(v)
        for k, v in user_symptoms_dict.items()
    }

    # ============================================================
    # BUILD USER VECTOR
    # ============================================================

    user_vector = []

    for col in feature_columns:

        severity = normalized_input.get(
            col.strip().lower(),
            0
        )

        user_vector.append(severity)

    user_vector = np.array(user_vector).reshape(1, -1)

    # Normalize severity 0-5 → 0-1
    user_vector_norm = scaler.transform(user_vector)

    # ============================================================
    # ANN PREDICTION
    # ============================================================

    ann_probs = model.predict_proba(user_vector_norm)[0]

    model_classes = model.classes_

    # ============================================================
    # WEIGHTED FUZZY MATCHING
    # ============================================================

    similarity_scores = {}

    for disease in prototype_df[TARGET_COLUMN].unique():

        disease_rows = prototype_df[
            prototype_df[TARGET_COLUMN] == disease
        ]

        disease_vectors = disease_rows[
            feature_columns
        ].astype(float).values

        disease_vectors = disease_vectors / 5.0

        best_score = 0

        for disease_vector in disease_vectors:

            # ====================================================
            # WEIGHTED MATCH
            # ====================================================
            
            rarity_weights = np.array([
                symptom_rarity[col]
                for col in feature_columns
            ])

            weighted_match = (
                user_vector_norm[0]
                * disease_vector
                * rarity_weights
            )
            
            match_score = np.sum(weighted_match) / (np.sum(disease_vector) + 1e-8)
            
            # ====================================================
            # IMPORTANT SYMPTOM COVERAGE
            # ====================================================
            
            important_symptoms = disease_vector >= 0.6
            
            matched_important = np.sum(
                (user_vector_norm[0] >= 0.2) &
                important_symptoms
            )
            
            total_important = np.sum(important_symptoms)
            
            if total_important > 0:
            
                coverage_score = (
                    matched_important / total_important
                )
            
            else:
            
                coverage_score = 0
            
            # ====================================================
            # NEGATIVE PENALTY
            # ====================================================
            
            missing_penalty = np.sum(
                important_symptoms &
                (user_vector_norm[0] < 0.2)
            )
            
            missing_penalty = (
                missing_penalty / max(total_important, 1)
            )
            
            # ====================================================
            # COSINE SUPPORT
            # ====================================================
            
            cosine = cosine_similarity(
                user_vector_norm,
                disease_vector.reshape(1, -1)
            )[0][0]
            
            # ====================================================
            # FINAL FUZZY SCORE
            # ====================================================
            
            final_fuzzy = (
                0.30 * match_score +
                0.30 * coverage_score +
                0.30 * cosine -
                0.30 * missing_penalty
            )

            final_fuzzy = max(final_fuzzy, 0)

            if final_fuzzy > best_score:
                best_score = final_fuzzy

        similarity_scores[disease] = best_score

    # ============================================================
    # HYBRID COMBINATION
    # ============================================================

    results = []

    for idx, encoded_class in enumerate(model_classes):

        disease = label_encoder.inverse_transform(
            [encoded_class]
        )[0]

        ann_score = ann_probs[idx]

        fuzzy_score = similarity_scores.get(
            disease,
            0
        )

        # ========================================================
        # FUZZY DOMINANT HYBRID
        # ========================================================

        final_score = (
            0.1 * ann_score +
            0.9 * fuzzy_score
        )

        results.append({

            "disease": disease,

            "ann_score": round(
                float(ann_score), 4
            ),

            "fuzzy_score": round(
                float(fuzzy_score), 4
            ),

            "final_score": round(
                float(final_score), 4
            )
        })

    # ============================================================
    # SORT RESULTS
    # ============================================================

    results = sorted(
        results,
        key=lambda x: x["final_score"],
        reverse=True
    )
    max_score = max(
        r["final_score"]
        for r in results
    )

    for r in results:

        confidence = (
            r["final_score"] /
            (max_score + 1e-8)
        ) * 100

        confidence *= min(
            len(user_symptoms_dict) / 5,
            1
        )

        r["confidence"] = round(
            min(max(confidence, 0), 100),
            2
        )

    return results[:top_n]

# ============================================================
# SAMPLE TEST
# ============================================================

print("\nRunning Sample Prediction...\n")

sample_input = {
    "painful urination": 5,
    "blood in urine": 5,
    "urinary obstruction": 5,
    "urination difficulty": 5,
    "burning sensation": 4
}

results = hybrid_predict(sample_input)

for r in results:
    print(r)

print("\nTraining Pipeline Complete.")

Loading dataset...


FileNotFoundError: [Errno 2] No such file or directory: 'C:\\Users\\verni\\Desktop\\Ayurwell\\Code\\data\\Ayurveda.xlsx'

In [17]:
import os
import joblib
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import MinMaxScaler
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import classification_report

DATASET_PATH = "data/Ayurveda.xlsx"

MODEL_DIR = "model_files"

os.makedirs(MODEL_DIR, exist_ok=True)

print("Loading dataset...")

df = pd.read_excel(DATASET_PATH)

# clean column names
df.columns = df.columns.str.strip().str.lower()

print("\nDataset Loaded Successfully.")
print(f"Shape: {df.shape}")

print("\nColumns:")
print(df.columns.tolist())

# =========================================================
# DATASET STRUCTURE
# =========================================================

# First 7 columns are metadata
# Remaining columns are symptom severity columns

METADATA_COLUMNS = df.columns[:7].tolist()

TARGET_COLUMN = "disease_english"

feature_columns = df.columns[7:].tolist()

print("\nMetadata Columns:")
print(METADATA_COLUMNS)

print(f"\nTotal Symptoms: {len(feature_columns)}")



# =========================================================
# CREATE FEATURE MATRIX
# =========================================================

X = df[feature_columns].copy()

# convert safely to numeric
X = X.apply(pd.to_numeric, errors="coerce")

# replace NaN
X = X.fillna(0)

# clip severity between 0 and 5
X = X.clip(0, 5)

print("\nFeature matrix created.")

# =========================================================
# CREATE LABELS
# =========================================================

y = df[TARGET_COLUMN].astype(str)

print("\nTarget labels created.")
print(f"Unique Diseases: {y.nunique()}")

# =========================================================
# SAVE FEATURE COLUMNS
# =========================================================

joblib.dump(
    feature_columns,
    os.path.join(MODEL_DIR, "columns.pkl")
)

print("Saved columns.pkl")



# =========================================================
# SYMPTOM RARITY WEIGHTS
# =========================================================

print("\nCalculating symptom rarity weights...")

symptom_rarity = {}

total_diseases = y.nunique()

for symptom in feature_columns:

    diseases_with_symptom = df[
        df[symptom] > 0
    ][TARGET_COLUMN].nunique()

    rarity_score = np.log(
        (total_diseases + 1) /
        (diseases_with_symptom + 1)
    )

    symptom_rarity[symptom] = float(rarity_score)

joblib.dump(
    symptom_rarity,
    os.path.join(MODEL_DIR, "symptom_rarity.pkl")
)

print("Saved symptom_rarity.pkl")

# =========================================================
# CREATE DISEASE PROFILES
# =========================================================

print("\nCreating disease profiles...")

disease_profiles = {}

for disease in y.unique():

    disease_rows = df[
        df[TARGET_COLUMN] == disease
    ]

    profile = {}

    for symptom in feature_columns:

        max_weight = disease_rows[symptom].max()

        if max_weight > 0:

            profile[symptom] = float(max_weight)

    disease_profiles[disease] = profile

joblib.dump(
    disease_profiles,
    os.path.join(MODEL_DIR, "disease_profiles.pkl")
)

print("Saved disease_profiles.pkl")

# =========================================================
# SAVE METADATA
# =========================================================

metadata = {}

for _, row in df.iterrows():

    disease = row[TARGET_COLUMN]

    metadata[disease] = {

        "description": str(row["discription"]),

        "remedies": str(row["remedies"]),

        "diet": str(row["diet"]),

        "lifestyle": str(row["lifestyle"])
    }

joblib.dump(
    metadata,
    os.path.join(MODEL_DIR, "metadata.pkl")
)

print("Saved metadata.pkl")

print("\nReasoning-based training pipeline complete.")


print("\nExample Disease Profile:\n")

sample_disease = list(disease_profiles.keys())[0]

print(sample_disease)

print(disease_profiles[sample_disease])

Loading dataset...

Dataset Loaded Successfully.
Shape: (1095, 259)

Columns:
['disease_english', 'disease_ayurvedic', 'discription', 'symptom', 'remedies', 'diet', 'lifestyle', 'skin discolouration', 'shrivelling', 'burning sensation', 'mild burning sensation', 'intense burning sensation', 'redness', 'vesicles', 'pain', 'pus formation', 'deep tissue damage', 'fever', 'thirst', 'unconsciousness', 'slow healing', 'scarring', 'indigestion', 'diminished appetite', 'loss of appetite', 'loss of taste', 'salivation', 'sour eructation', 'heaviness in abdomen', 'lassitude', 'heaviness in body', 'retention of gases', 'constipation', 'loose motions', 'abdominal distension', 'severe abdominal pain', 'retention of faeces', 'flatus', 'eructation', 'undigested food', 'obstruction', 'slimy stool', 'foul-smelling stool', 'frequent motions', 'lightness in body', 'chronic diarrhea', 'blood in stool', 'gripping abdominal pain', 'swelling of anus', 'rectal prolapse', 'mucoid stool', 'straining', 'acute pa

In [18]:
# =========================================================
# HYBRID PREDICTION V2
# =========================================================

def hybrid_predict_v2(user_symptoms):

    results = []

    # =====================================================
    # NORMALIZE INPUT
    # =====================================================

    normalized_input = {}

    for symptom, severity in user_symptoms.items():

        symptom = symptom.strip().lower()

        severity = float(severity)

        severity = max(0, min(severity, 5))

        normalized_input[symptom] = severity

    # =====================================================
    # SCORE EACH DISEASE
    # =====================================================

    for disease, profile in disease_profiles.items():

        total_score = 0

        matched_score = 0

        penalty_score = 0

        matched_symptoms = []

        hallmark_total = 0

        hallmark_matched = 0

        # =================================================
        # SYMPTOM MATCHING
        # =================================================

        for symptom, disease_weight in profile.items():

            rarity = symptom_rarity.get(
                symptom,
                1
            )

            user_severity = normalized_input.get(
                symptom,
                0
            )

            # =============================================
            # SEVERITY ALIGNMENT
            # =============================================

            alignment = (
                1 -
                abs(user_severity - disease_weight) / 5
            )

            alignment = max(alignment, 0)

            # =============================================
            # CORE MATCH SCORE
            # =============================================

            symptom_score = (
                user_severity *
                disease_weight *
                rarity *
                alignment
            )
            # =============================================
            # HALLMARK BOOST
            # =============================================

            if disease_weight >= 4:

                hallmark_total += 1

                symptom_score *= 1.3

                if user_severity > 0:

                    hallmark_matched += 1
            # =============================================
            # CONTRADICTION PENALTY
            # =============================================

            if disease_weight >= 4 and user_severity == 0:

                penalty_score += (
                    disease_weight *
                    rarity * 1.2
                )
            # =============================================
            # POSITIVE MATCH
            # =============================================

            if user_severity > 0:

                matched_score += symptom_score

                matched_symptoms.append({

                    "symptom": symptom,

                    "user_severity": user_severity,

                    "disease_weight": disease_weight,

                    "score": round(symptom_score, 2)
                })
         # =================================================
         # HALLMARK COVERAGE
         # =================================================

        if hallmark_total > 0:

             coverage_score = (
                 hallmark_matched /
                 hallmark_total
             )

        else:

             coverage_score = 0
        # =================================================
        # FINAL SCORE
        # =================================================

        symptom_match_count = len(matched_symptoms)
        
        match_factor = min(
            symptom_match_count / 3,
            1
        )
        
        total_score = (
            matched_score *
            (1 + coverage_score) *
            match_factor
        ) - penalty_score

        total_score = max(total_score, 0)

        results.append({

            "disease": disease,

            "score": round(total_score, 2),

            "coverage": round(coverage_score, 2),

            "matched_symptoms": matched_symptoms
        })
     # =====================================================
     # SORT RESULTS
     # =====================================================
    results = sorted(
         results,
         key=lambda x: x["score"],
         reverse=True
     )
    # =====================================================
    # CONFIDENCE CALCULATION
    # =====================================================
    if not results:

        return {

            "top_predictions": [],

            "follow_up_needed": False,

            "follow_up_context": {}
        }

    max_score = results[0]["score"]
    
    for result in results:
        confidence = (
              result["score"] /
              (max_score + 1e-8)
          ) * 100
        # ================================================
        # LOW INPUT PENALTY
        # ================================================
        symptom_factor = min(
            len(normalized_input) / 5,
            1
        )
        confidence *= symptom_factor
        confidence = max(
            0,
            min(confidence, 100)
        )
        result["confidence"] = round(
            confidence,
            2
        )
    # =====================================================
    # AMBIGUITY DETECTION
    # =====================================================
    follow_up_needed = False
    follow_up_context = {}
    if len(results) >= 2:
            
        top_1 = results[0]
        top_2 = results[1]
        score_gap = (
            top_1["confidence"] -
            top_2["confidence"]
        )
        if score_gap < 15:
            follow_up_needed = True
        disease_1_profile = disease_profiles[
            top_1["disease"]
        ]
        disease_2_profile = disease_profiles[
            top_2["disease"]
        ]
        candidate_questions = []
        all_symptoms = set(
            disease_1_profile.keys()
        ).union(
            disease_2_profile.keys()
        )
        for symptom in all_symptoms:
            w1 = disease_1_profile.get(
                symptom,
                0
            )
            w2 = disease_2_profile.get(
                symptom,
                0
            )
            difference = abs(w1 - w2)
            if (
                difference >= 3 and
                symptom not in normalized_input
            ):
                candidate_questions.append({
                    "symptom": symptom,
                    "difference": difference
                })
        candidate_questions = sorted(
            candidate_questions,
            key=lambda x: x["difference"],
            reverse=True
        )
        follow_up_context = {
        
            "top_disease": top_1["disease"],

            "second_disease": top_2["disease"],

            "questions": candidate_questions[:5]
        }
    return {
    
        "top_predictions": results[:5],
        "follow_up_needed": follow_up_needed,
        "follow_up_context": follow_up_context
    }

test_input = {

    "burning sensation": 5,

    "vesicles": 4,

    "pain": 4,

    "skin discolouration": 3
}

result = hybrid_predict_v2(test_input)

print(result)  



{'top_predictions': [{'disease': 'Burns and scalds', 'score': 162.18, 'coverage': 0.38, 'matched_symptoms': [{'symptom': 'skin discolouration', 'user_severity': 3.0, 'disease_weight': 3.0, 'score': 35.15}, {'symptom': 'burning sensation', 'user_severity': 5.0, 'disease_weight': 5.0, 'score': 53.19}, {'symptom': 'vesicles', 'user_severity': 4.0, 'disease_weight': 4.0, 'score': 81.23}, {'symptom': 'pain', 'user_severity': 4.0, 'disease_weight': 4.0, 'score': 20.35}], 'confidence': 80.0}, {'disease': 'Eyelid pustule', 'score': 98.06, 'coverage': 1.0, 'matched_symptoms': [{'symptom': 'burning sensation', 'user_severity': 5.0, 'disease_weight': 5.0, 'score': 53.19}, {'symptom': 'pain', 'user_severity': 4.0, 'disease_weight': 4.0, 'score': 20.35}], 'confidence': 48.37}, {'disease': 'Gout', 'score': 72.53, 'coverage': 1.0, 'matched_symptoms': [{'symptom': 'burning sensation', 'user_severity': 5.0, 'disease_weight': 4.0, 'score': 34.04}, {'symptom': 'pain', 'user_severity': 4.0, 'disease_weigh